# TalentFlow — Colab AI server cho module phân tích transcript phỏng vấn

Notebook này mở 1 server AI (Ollama) chạy trên GPU free của Colab, expose ra 1 URL public để backend TalentFlow (chạy trên máy bạn) gọi sang xử lý.

**Cách dùng mỗi lần cần phân tích transcript:**
1. Vào `Runtime > Change runtime type` chọn **T4 GPU**, Save.
2. `Runtime > Run all` (chạy hết các cell từ trên xuống).
3. Đợi tới cell cuối, copy dòng `COLAB_LLM_ENDPOINT_URL=...` nó in ra.
4. Dán đúng dòng đó vào file `backend/.env` trên máy bạn (đè lên dòng `COLAB_LLM_ENDPOINT_URL` cũ).
5. Restart backend trên máy bạn.

Lưu ý: mỗi lần chạy lại notebook này, URL sẽ đổi khác — phải lặp lại bước 3-5. Phiên Colab free tự ngắt sau ~90 phút không thao tác.

## Bước 1 — Kiểm tra GPU
Phải thấy tên GPU (vd Tesla T4) ở dưới. Nếu báo lỗi `command not found`, vào `Runtime > Change runtime type` chọn GPU rồi chạy lại từ đầu.

In [ ]:
!nvidia-smi

## Bước 2 — Cài Ollama
`zstd` là gói bị thiếu sẵn trên máy Colab mà script cài Ollama cần — cài trước để tránh lỗi `FileNotFoundError: ollama`.

In [ ]:
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

## Bước 3 — Khởi động Ollama
`OLLAMA_ORIGINS=*` là bắt buộc — thiếu dòng này Ollama sẽ trả 403 rỗng cho mọi request đi qua tunnel (nó chặn theo header `Host` để chống DNS rebinding, mặc định chỉ tin `localhost`).

In [ ]:
import os, subprocess, time

os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_HOST"] = "0.0.0.0"
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("Ollama server đã khởi động.")

## Bước 4 — Tải 3 model mạnh nhất, free, phù hợp phân tích văn bản
`qwen2.5:14b` mạnh nhất trong 3 (giỏi tiếng Việt + tuân thủ JSON tốt), `llama3.1:8b` và `gemma2:9b` làm 2 nguồn đối chiếu thêm. Lần đầu tải mất vài phút (tải ~19GB tổng cộng).

In [ ]:
!ollama pull qwen2.5:14b
!ollama pull llama3.1:8b
!ollama pull gemma2:9b

## Bước 4b — Cài STT (faster-whisper) để trích xuất giọng nói từ bản ghi âm
Dùng cho tính năng "Ghi âm phỏng vấn" trên TalentFlow: nhận file audio, trả về text. Chạy model `medium` (đủ tốt cho tiếng Việt, cân bằng tốc độ/VRAM trên GPU T4 free) — nếu thấy độ chính xác chưa đủ, đổi `"medium"` thành `"large-v3"` ở ô dưới (vẫn free, chỉ chậm hơn).

In [ ]:
!pip install -q faster-whisper flask


In [ ]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel("medium", device="cuda", compute_type="float16")
print("Đã tải xong model Whisper (medium).")


## Bước 4c — Mở server nhận audio, trả về text
Chạy ở port riêng (8001), tách biệt hoàn toàn với Ollama (11434) — ghi âm và 3 model phân tích chạy song song không ảnh hưởng nhau.

In [ ]:
import threading, traceback, subprocess
import numpy as np
from flask import Flask, jsonify, request as flask_request

stt_app = Flask(__name__)

@stt_app.route("/transcribe", methods=["POST"])
def transcribe():
    try:
        audio_file = flask_request.files["audio"]
        audio_path = "/tmp/talentflow_recording.webm"
        audio_file.save(audio_path)
        # Decode qua binary ffmpeg có sẵn trên Colab ra PCM16 mono 16kHz, thay vì để
        # faster-whisper tự decode bằng PyAV - bản av cài sẵn trên Colab hay bị lệch
        # version với cái faster-whisper cần, gây lỗi "unexpected keyword argument
        # 'metadata_errors'". Đưa thẳng mảng số cho transcribe() bỏ qua PyAV luôn.
        result = subprocess.run(
            ["ffmpeg", "-y", "-i", audio_path, "-f", "s16le", "-ac", "1", "-ar", "16000", "-"],
            capture_output=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"ffmpeg lỗi: {result.stderr.decode(errors='ignore')[-500:]}")
        audio_array = np.frombuffer(result.stdout, dtype=np.int16).astype(np.float32) / 32768.0
        segments, _info = whisper_model.transcribe(audio_array, language="vi")
        text = " ".join(segment.text.strip() for segment in segments)
        return jsonify({"text": text})
    except Exception as exc:
        # In lỗi thật ra ngay ô này (thay vì chỉ thấy trang 500 chung chung phía backend)
        # và trả kèm message trong JSON để dễ debug từ xa.
        traceback.print_exc()
        return jsonify({"error": str(exc)}), 500

threading.Thread(target=lambda: stt_app.run(host="0.0.0.0", port=8001), daemon=True).start()
time.sleep(3)
print("STT server đã khởi động ở port 8001.")


## Bước 4d — Mở tunnel public riêng cho STT
Tunnel LLM ở Bước 5 (port 11434) giữ nguyên, không đổi — đây là tunnel **thứ hai**, riêng cho STT (port 8001).

In [ ]:
!nohup ./cloudflared tunnel --url http://localhost:8001 > cf_tunnel_stt.log 2>&1 &
time.sleep(8)
!cat cf_tunnel_stt.log


Nếu chưa thấy dòng `https://....trycloudflare.com` ở trên, chạy lại ô dưới đây sau vài giây.

In [ ]:
import re

with open("cf_tunnel_stt.log") as f:
    stt_log_content = f.read()
stt_match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", stt_log_content)
stt_tunnel_url = stt_match.group(0) if stt_match else None
print("STT Tunnel URL:", stt_tunnel_url or "CHƯA THẤY — chạy lại ô này sau vài giây")


## Bước 5 — Mở tunnel public (Cloudflare Tunnel)
Dùng `cloudflared` thay vì `localtunnel` vì ổn định hơn nhiều (localtunnel hay bị 502/timeout ngẫu nhiên, đã kiểm chứng thực tế).

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:11434 > cf_tunnel.log 2>&1 &
time.sleep(8)
!cat cf_tunnel.log

Nếu ở trên chưa thấy dòng `https://....trycloudflare.com`, chạy lại ô dưới đây (tunnel có thể cần thêm vài giây để khởi tạo).

In [ ]:
import re

with open("cf_tunnel.log") as f:
    log_content = f.read()
match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", log_content)
tunnel_url = match.group(0) if match else None
print("Tunnel URL:", tunnel_url or "CHƯA THẤY — chạy lại ô này sau vài giây")

## Bước 6 — Tự kiểm tra kết nối (gọi thử 1 câu hỏi thật qua tunnel)
Nếu ra `HTTP 200` kèm nội dung JSON là xong, có thể dùng được.

In [ ]:
import requests

if tunnel_url:
    try:
        response = requests.post(
            f"{tunnel_url}/v1/chat/completions",
            json={
                "model": "qwen2.5:14b",
                "messages": [{"role": "user", "content": "Trả lời bằng JSON: {\"ok\": true}"}],
                "response_format": {"type": "json_object"},
                "max_tokens": 50,
            },
            timeout=90,
        )
        print("HTTP", response.status_code)
        print(response.text[:500])
    except Exception as exc:
        print("Lỗi khi gọi thử:", exc)
else:
    print("Chưa có tunnel URL — quay lại Bước 5.")

## Bước 7 — Giá trị cần dán vào `backend/.env`
Copy đúng dòng in ra dưới đây, dán đè vào `backend/.env` trên máy bạn, rồi restart backend.

In [ ]:
if tunnel_url:
    print(f"COLAB_LLM_ENDPOINT_URL={tunnel_url}/v1/chat/completions")
    print("COLAB_LLM_MODEL=qwen2.5:14b")
else:
    print("Chưa có tunnel URL — chạy lại Bước 5 rồi quay lại đây.")

if stt_tunnel_url:
    print(f"COLAB_STT_ENDPOINT_URL={stt_tunnel_url}/transcribe")
else:
    print("Chưa có STT tunnel URL — chạy lại Bước 4d rồi quay lại đây.")


---
**Giữ tab này luôn mở** trong lúc dùng tính năng phân tích transcript trên TalentFlow — Colab ngắt phiên là tunnel chết theo, phải chạy lại từ Bước 1.